In [1]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
import os
%matplotlib qt

frame_num = 370
cam = 0
image_path = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/'
dict_path  = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/dict/frames_model.pkl'
path_output = 'D:/Documents/gaussian_model_output/'
interest_point_h5_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evalutation'


image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov128_2023_08_09_60ms/'
dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/dict/frames_model_evaluation.pkl'
# path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

# image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
# dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model_evaluation.pkl'
# path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

model_name = 'fly_features_compare'
file_name = 'fly_model'


model_name = 'fly_features_dense'
file_name = 'fly_model'

idx_iter = 0
model_name = 'fly_yaw_sweep'
file_name = f'fly_model_scale_iter{idx_iter}'
# model_name = 'fly_features_3cam'
# file_name = 'fly_model_aa'
iteration = 1000
frame0 = 1650#1620
input_dir = f'{path_output}/{model_name}'
num_iter = 2


# with open(f'{input_dir}/results/{frame0}/{file_name}_results.pkl', 'rb') as handle:
#     output_angles_weights = pickle.load(handle)
    

with open(dict_path,'rb') as f:
    frames = pickle.load(f)

# wakk = [output_angles_weights['weights'][idx] for idx in range(idx_iter,len(output_angles_weights['weights']),num_iter)]
# weights= {}
# weights['weights'] = wakk



input_dir_ini = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation'

with open(f'{input_dir_ini}/nominal_initial_angles.pkl', 'rb') as handle:
    nominal_initial_angles = pickle.load(handle)



# weights = output_angles_weights['weights'][0]


In [46]:
mov_frame = 'mov_59_frame_2016'
mov = int(mov_frame.split('_')[1]) 
frame = int(mov_frame.split('_')[3]) 


path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}/wing1_gt_points_frame{frame}.pkl'
with open(path,'rb') as f:
    ini_angles1 = pickle.load(f)
cam1 = np.vstack(ini_angles1['cam1'])
plt.scatter(cam1[:,0],cam1[:,1])
path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}/wing2_gt_points_frame{frame}.pkl'
with open(path,'rb') as f:
    ini_angles2 = pickle.load(f)
cam1 = np.vstack(ini_angles2['cam1'])
plt.scatter(cam1[:,0],cam1[:,1])

In [21]:
def open_file(path,ang_dict = {}):
    with open(path,'rb') as f:
        ini_angles = pickle.load(f)

    ang_dict = {ang_name: angles for ang_name,angles in ini_angles.items()}

    return ang_dict

frame = 223

input_dir = f'{path_output}/{model_name}'
path = f'D:/Documents/gaussian_model_output/fly_yaw_sweep/{frame}/initial/'
dir_names = [name for name in os.listdir(path) if os.path.isfile(os.path.join(path, name))]
ini_angles = {idx:open_file(f'{path}/{dir}') for idx,dir in enumerate(dir_names)}

bod = np.vstack([ini_angles[idx]['body_angles'] for idx in range(len(ini_angles))])

bod


array([[ 13., -40.,  -4.],
       [ 13., -40.,  -4.],
       [ -7., -40.,  -4.],
       [ -2., -40.,  -4.],
       [  3., -40.,  -4.],
       [  8., -40.,  -4.],
       [ 18., -40.,  -4.],
       [ 23., -40.,  -4.],
       [ 28., -40.,  -4.]])

In [38]:
list(nominal_initial_angles.keys())[1]

'mov_36_frame_1031'

In [6]:
from Evaluation import Evaluation





letedict = {'num_of_bins' : 10,'perc_wing_for_le' : 1, 'wing_length_snip':0.2}


iteration = 1000
idx_iter = 9
mov_name = list(nominal_initial_angles.keys())[6]
all_frames = []


def load_frame_all_sweep(idx_iter,mov_name,iteration,letedict,frames):
    mov = int(mov_name.split('_')[1]) 
    frame0 = int(mov_name.split('_')[3]) 
    image_path =  f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{mov}_2023_08_09_60ms/'
    file_name = f'fly_model_scale_iter{idx_iter}'
    interest_points_path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}'

    with open(f'{input_dir}/results/{frame0}/{file_name}_results.pkl', 'rb') as handle:
        output_angles_weights = pickle.load(handle)

    frame_eval = Evaluation(interest_points_path,image_path,frame0,input_dir,output_angles_weights,frame0,iteration,file_name,letedict = letedict,frames_dict = frames)
    for source_attr, target_attr, output_attr in frame_eval.projection_tasks:
        frame_eval.get_projected_and_store(frame_eval, source_attr, target_attr, output_attr)
    return frame_eval



all_frames = [load_frame_all_sweep(idx_iter,mov_name,iteration,letedict,frames) for idx_iter in range(9)]
[frame.calculate_error() for frame in all_frames]  






[None, None, None, None, None, None, None, None, None]

In [7]:

frame = all_frames[0]

fig = go.Figure()
Plotters.scatter3d(fig,frame.right_wing_boundary,'crimson',4,'right wing boundary',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,frame.right_wing,'red',2,'right wing',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.interest_right_wing_boundry,'magenta',6,'tagged points',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.bound_on_interest_rw,'black',4,'fitted boundary',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing_boundary,'#1f77b4',3,'left wing boundary',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,frame.left_wing,'blue',2,'left wing',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.interest_left_wing_boundry,'#17becf',6,'tagged points',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.bound_on_interest_lw,'black',4,'fitted boundary',show_colorbar = False)

Plotters.scatter3d(fig,frame.body,'green',4,'body',show_colorbar = False,mode=' markers')

fig.show()

In [ ]:
        self.error2d_gt_on_boundary_to_gt = self.calculate_repreojection_error(self.att_to_calc_interest) # distance between gt on banudary to baundary
        self.error3d_gt_on_boundary_to_gt = self.calculate_3d_dist(self.att_to_calc_interest)
        self.error2d_boundary_on_gt_to_boundary= self.calculate_repreojection_error(self.att_to_calc_bound) # distance between boundary on gt to gt
        self.error3d_boundary_on_gt_to_boundary = self.calculate_3d_dist(self.att_to_calc_bound)
        

In [87]:
all_frames[0].interest_points_3d.shape

(26, 3)

In [89]:
all_frames[0].error2d_boundary_on_gt_to_boundary.shape

(132,)

In [10]:
plt.hist(all_frames[0].error2d_boundary_on_gt_to_boundary, bins = 50)

(array([91., 47., 24., 13.,  6.,  1.,  3.,  2.,  3.,  1.,  2.,  0.,  1.,
         2.,  0.,  1.,  0.,  0.,  1.,  0.,  0.,  1.,  0.,  1.,  0.,  0.,
         0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
         0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.]),
 array([ 0.05670082,  0.84691762,  1.63713442,  2.42735122,  3.21756802,
         4.00778482,  4.79800162,  5.58821841,  6.37843521,  7.16865201,
         7.95886881,  8.74908561,  9.53930241, 10.32951921, 11.11973601,
        11.90995281, 12.70016961, 13.49038641, 14.28060321, 15.07082   ,
        15.8610368 , 16.6512536 , 17.4414704 , 18.2316872 , 19.021904  ,
        19.8121208 , 20.6023376 , 21.3925544 , 22.1827712 , 22.972988  ,
        23.7632048 , 24.55342159, 25.34363839, 26.13385519, 26.92407199,
        27.71428879, 28.50450559, 29.29472239, 30.08493919, 30.87515599,
        31.66537279, 32.45558959, 33.24580639, 34.03602318, 34.82623998,
        35.61645678, 36.40667358, 37.19689038, 37.98710718,

In [8]:
all_sweep = np.hstack([frame.error2d_boundary_on_gt_to_boundary for frame in all_frames]) 
np.mean(all_sweep)

[np.mean(frame.error2d_boundary_on_gt_to_boundary) for frame in all_frames]

# [np.std(frame.error2d_boundary_on_gt_to_boundary) for frame in all_frames]

[2.443670211029306,
 3.070390724660352,
 1.9990445515024118,
 2.893375302823638,
 2.6046316644309417,
 3.5159787981874455,
 2.4953215009426386,
 3.442994073059234,
 1.8345800745543712]

In [15]:
from Evaluation import Evaluation
mov_frame = list(nominal_initial_angles.keys())[9]
mov = int(mov_frame.split('_')[1]) 
frame0 = int(mov_frame.split('_')[3]) 



def calculate_error(att_to_calc_interest,att_to_calc_bound):
    error2d_gt_on_boundary_to_gt = frame.calculate_repreojection_error(att_to_calc_interest) # distance between gt on banudary to baundary
    error3d_gt_on_boundary_to_gt = frame.calculate_3d_dist(att_to_calc_interest)
    error2d_boundary_on_gt_to_boundary= frame.calculate_repreojection_error(att_to_calc_bound) # distance between boundary on gt to gt
    error3d_boundary_on_gt_to_boundary = frame.calculate_3d_dist(att_to_calc_bound)
    return error2d_gt_on_boundary_to_gt,error3d_gt_on_boundary_to_gt,error2d_boundary_on_gt_to_boundary,error3d_boundary_on_gt_to_boundary



interest_points_path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}'
image_path =  f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{mov}_2023_08_09_60ms/'

letedict = {'num_of_bins' : 20,'perc_wing_for_le' : 1, 'wing_length_snip':0.27}
projection_tasks = [
    ("interest_right_wing_boundry", "right_wing_boundary", "interest_on_bound_rw"),
    ("interest_left_wing_boundry", "left_wing_boundary", "interest_on_bound_lw"),
    ("right_wing_boundary", "interest_right_wing_boundry", "bound_on_interest_rw"),
    ("left_wing_boundary", "interest_left_wing_boundry", "bound_on_interest_lw"),
]
att_to_calc_interest = [
    "interest_on_bound_rw", "interest_on_bound_lw", "interest_right_wing_boundry","interest_left_wing_boundry",    
]

att_to_calc_bound = [
    "bound_on_interest_rw", "bound_on_interest_lw", "right_wing_boundary","left_wing_boundary",    
]

all_range_error2d_gt_on_boundary_to_gt = []
all_range_error3d_gt_on_boundary_to_gt = []
all_range_error2d_boundary_on_gt_to_boundary = []
all_range_error3d_boundary_on_gt_to_boundary = []

iteration = 1000
for idx_iter in range(8):



    file_name = f'fly_model_scale_iter{idx_iter}'
    with open(f'{input_dir}/results/{frame0}/{file_name}_results.pkl', 'rb') as handle:
        output_angles_weights = pickle.load(handle)

    frame = Evaluation(interest_points_path,image_path,frame0,input_dir,output_angles_weights,frame0,iteration,file_name,letedict = letedict,frames_dict = frames)
    for source_attr, target_attr, output_attr in projection_tasks:
        frame.get_projected_and_store(frame, source_attr, target_attr, output_attr)
    error2d_gt_on_boundary_to_gt,error3d_gt_on_boundary_to_gt,error2d_boundary_on_gt_to_boundary,error3d_boundary_on_gt_to_boundary = calculate_error(att_to_calc_interest,att_to_calc_bound)

    all_range_error2d_boundary_on_gt_to_boundary.append(error2d_boundary_on_gt_to_boundary)
    all_range_error3d_boundary_on_gt_to_boundary.append(error3d_boundary_on_gt_to_boundary)
    all_range_error2d_gt_on_boundary_to_gt.append(error2d_gt_on_boundary_to_gt)
    all_range_error3d_gt_on_boundary_to_gt.append(error3d_gt_on_boundary_to_gt)



fig = go.Figure()
Plotters.scatter3d(fig,frame.right_wing_boundary,'crimson',4,'right wing boundary',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,frame.right_wing,'red',2,'right wing',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.interest_right_wing_boundry,'magenta',6,'tagged points',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.bound_on_interest_rw,'black',4,'fitted boundary',show_colorbar = False)

# Plotters.scatter3d(fig,frame.interest_right_wing_boundry[a],'black',4,'right wing',show_colorbar = False)
# Plotters.scatter3d(fig,interest_on_bound3,'black',4,'right wing',show_colorbar = False)
# Plotters.scatter3d(fig,frame.right_wing_boundary,'black',4,'right wing',show_colorbar = False)


Plotters.scatter3d(fig,frame.left_wing_boundary,'#1f77b4',3,'left wing boundary',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,frame.left_wing,'blue',2,'left wing',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.interest_left_wing_boundry,'#17becf',6,'tagged points',show_colorbar = False,mode=' markers')
Plotters.scatter3d(fig,frame.bound_on_interest_lw,'black',4,'fitted boundary',show_colorbar = False)

Plotters.scatter3d(fig,frame.body,'green',4,'body',show_colorbar = False,mode=' markers')

fig.show()


output_path = 'G:/My Drive/Research/seminar/evaluation_1000iter.html'

fig.show()
fig.write_html(output_path)
print(f"Saved animation to: {output_path}")

Saved animation to: G:/My Drive/Research/seminar/evaluation_1000iter.html


In [29]:
fig,axs = plt.subplots(2,2)
bins_2d = 100
bins3d = 100

def get_indices_top97(data_to_hist, percent = 0.97):
    indices = [int(data_to_hist[idx].shape[0]*percent) for idx in range(len(data_to_hist))]
    top_97 = np.hstack([ np.sort(np.hstack(data_to_hist[idx]))[:idx_finish] for idx,idx_finish in enumerate(indices)])
    return top_97
    # all_range_error2d_gt_on_boundary_to_gt.append(error2d_boundary_on_gt_to_boundary)
    # all_range_error3d_gt_on_boundary_to_gt.append(error3d_boundary_on_gt_to_boundary)
    # all_range_error2d_boundary_on_gt_to_boundary.append(error2d_gt_on_boundary_to_boundary)
    # all_range_error3d_boundary_on_gt_to_boundary.append(error3d_gt_on_boundary_to_boundary)

frame_for_hist = all_frames[5]
axs[0,0].hist(get_indices_top97([frame_for_hist.error2d_boundary_on_gt_to_boundary]), bins = bins_2d),
axs[0,0].set_title('Boundary on Gt to boundary'),
axs[0,1].hist(get_indices_top97([frame_for_hist.error2d_gt_on_boundary_to_gt]), bins = bins_2d),
axs[0,1].set_title('Gt on boundary to Gt')
axs[0,0].set_xlabel('pixel'),
axs[0,1].set_xlabel('pixel'),
fig.suptitle("Reprojection and 3D distance error")

axs[1,0].hist(get_indices_top97([frame_for_hist.error3d_boundary_on_gt_to_boundary]), bins = bins3d),
axs[1,1].hist(get_indices_top97([frame_for_hist.error3d_gt_on_boundary_to_gt]), bins = bins3d),
axs[1,0].set_title('Boundary on Gt to boundary'),
axs[1,1].set_title('Gt on boundary to Gt')
axs[1,0].set_xlabel('mm'),
axs[1,1].set_xlabel('mm'),

plt.tight_layout()


[np.mean(get_indices_top97(forerror, percent=0.99)) for forerror in [all_range_error2d_boundary_on_gt_to_boundary,all_range_error2d_gt_on_boundary_to_gt,all_range_error3d_boundary_on_gt_to_boundary,
                                                       all_range_error3d_gt_on_boundary_to_gt]]

NameError: name 'all_range_error2d_boundary_on_gt_to_boundary' is not defined

In [41]:
frames_num = [int(frame.split('_')[3]) for frame in list(nominal_initial_angles.keys())]

In [30]:
import plotly.graph_objects as go
import numpy as np


color_list = ['lime','crimson','magenta','magenta','dodgerblue','blue','blue','black','orange']
name_list = ['body','right wing','right wing le','right wing te','left wing','left wing le','left wing te','Ground truth','gaussian points']
size_list = [2,2,4,4,2,4,4,5,5]
framestart = 1031
frame_end = 1032
frames = range(framestart, frame_end)

output_path = f'{path_output}/{model_name}/animated_plot.html'

xyz_all_frames = np.vstack([frame.xyz_rotated for frame in all_frames])

# === HELPERS ===

def create_scatter3d(xyz, color,name,size = 2):
    """Create a single 3D scatter trace for a specific part."""
    return go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode="markers",
        name = name,
        marker=dict(size=size, opacity=1, color=color, colorscale='gray'),
    )


def get_global_bounds(xyz_list):
    """Compute global min and max coordinates over all frames for consistent axis scaling."""
    return np.min(xyz_list, axis=0), np.max(xyz_list, axis=0)


def create_frame(parts_list, color_list,size_list, frame_name,name_list):
    """Create one animation frame with all parts for a given timestep."""
    data = [
        create_scatter3d(part, color,name,size)
        for part, color,size,name in zip(parts_list, color_list, size_list,name_list)
    ]
    return go.Frame(data=data, name=frame_name)


def create_play_pause_buttons():
    """Return Play/Pause button definitions for animation."""
    return [
        {
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ]

def create_slider(frame_nums):
    """Create a slider using actual frame numbers."""
    return [
        {
            "active": 0,
            "steps": [
                {
                    "args": [[str(i)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                    "label": str(frame_num),
                    "method": "animate",
                }
                for i, frame_num in enumerate(frame_nums)
            ],
        }
    ]


# === MAIN FUNCTION ===

def create_3d_animation(frames_list, color_list,xyz_all_frames,size_list,name_list):
    """Build and show the 3D animation."""
    # frame_nums = [f.frame_num for f in frames_list[framestart - frame0:frame_end - frame0]]
    frame_nums = [frame.frame_num for frame in all_frames]
    min_xyz, max_xyz = get_global_bounds(xyz_all_frames)

    # Initial frame data
    intial_parts = [frames_list[0].body,frames_list[0].right_wing,frames_list[0].right_wing_le,frames_list[0].right_wing_te,frames_list[0].left_wing,frames_list[0].left_wing_le,frames_list[0].left_wing_te, frames_list[0].rotated_points_3d]
    initial_data = [
        create_scatter3d(part, color,name,size)
        for  part,color,size,name in zip(intial_parts, color_list,size_list,name_list)
    ]
    bounding_box_trace = go.Scatter3d(

    mode='markers',
    marker=dict(size=0.1, color='rgba(0,0,0,0)'),
    showlegend=False
    )
    initial_data.append(bounding_box_trace)
    # Create frames for animation
    frames_data = [
        create_frame([xyz_frame.body,xyz_frame.right_wing,xyz_frame.right_wing_le,xyz_frame.right_wing_te,xyz_frame.left_wing,xyz_frame.left_wing_le,xyz_frame.left_wing_te, xyz_frame.rotated_points_3d], color_list,size_list, str(i),name_list)
        for i, xyz_frame in enumerate(frames_list)
    ]

    # Build full figure
    fig = go.Figure(
        data=initial_data,
        layout=go.Layout(
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            updatemenus=create_play_pause_buttons(),
            sliders=create_slider(frame_nums),
        ),
        frames=frames_data,
    )

    fig.show()
    fig.write_html(output_path)
    print(f"Saved animation to: {output_path}")


create_3d_animation(all_frames, color_list,xyz_all_frames,size_list,name_list)




    # point_3d_per_frame.append(np.vstack(points_3d))
    # gaussians_interest_points.append(gaussian_points)

Saved animation to: D:/Documents/gaussian_model_output//fly_nominal_data_update/animated_plot.html


In [56]:
all_frames[0].frame_num

1620

In [527]:
    import itertools
    yaw_grid = np.hstack(np.arange(0,360,30))
    roll_grid = np.hstack(np.arange(-30,30,10))
    psi_grid = np.hstack((np.arange(-160,0,30)))
    phi_grid = np.hstack((np.arange(-90,90,30)))

    roll_yaw = list(itertools.product(yaw_grid,roll_grid))
    psi_phi = list(itertools.product(psi_grid,phi_grid))
    # roll_yaw = roll_grid
    # pitch_grid = np.hstack((0.0,np.arange(-20,0,5),np.arange(5,20,5)))
    roll_yaw = roll_yaw + psi_phi

In [530]:
108-36

72

In [54]:
frame = 370
color = frames_list[frame - frame0].color
idx_part = frames_list[frame - frame0].idx_parts


grayscale = (color[:,0] - color[:,0].min()) / (color[:,0].max() - color[:,0].min())

opacity = frames_list[frame - frame0].opacity * grayscale
# grayscale = grayscale[grayscale <1]

frame = frames_list[frame-frame0]
fig = go.Figure()
Plotters.scatter3d(fig,frame.body,'green',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing,opacity[idx_part[1]],2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing,opacity[idx_part[2]],2,'left wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.rotated_points_3d[7:8,:],'blue',5,'interest',show_colorbar = False)
# Plotters.scatter3d(fig,frames_list[frame-frame0].gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_te,'cyan',4,'te',show_colorbar = False)

Plotters.scatter3d(fig,frame.left_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing_te,'cyan',4,'te',show_colorbar = False)


t = np.linspace(-0.001, 0.0015, 100)  # Small range since your data seems very small-scale

r_line_points = frame.right_wing_origin + t[:, np.newaxis] * frame.right_wing_span
l_line_points = frame.left_wing_origin + t[:, np.newaxis] * frame.left_wing_span
idx_closest = np.unique([np.argsort(frame.dist_points(point,frame.body))[0:10] for point in r_line_points])
# idx_origin = np.argmin(np.dot(frame.body[idx_closest,:],frame.right_wing_direction))
idx_origin = np.argmax([np.min(np.dot(frame.body[idx_closest,:],r_line_points.T)) for idx_closest in idx_closest])
idx_origin


Plotters.scatter3d(fig,r_line_points,'orange',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_line_points,'orange',3,'wing',show_colorbar = False)
# Plotters.scatter3d(fig,np.atleast_2d(frame.body[idx_closest[idx_origin],:]),'black',10,'wing',show_colorbar = False)

# Plotters.scatter3d(fig,frame.gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
# Plotters.scatter3d(fig,frame.body_interest_gaussian,'orange',5,'interest gauss',show_colorbar = False)
# dot_on_le = np.dot(-frame.right_wing_direction,frame.right_wing_le.T)
# length = (np.max(dot_on_le) - np.min(dot_on_le))
# lt20p = frame.right_wing_le[(dot_on_le  - np.min(dot_on_le))> length*0.1,:]
# Plotters.scatter3d(fig,frame.rotated_points_3d[8:16],'orange',3,'interest gauss',show_colorbar = False)
# Plotters.scatter3d(fig, np.vstack((np.mean(frame.bottom,axis = 0) - frame.xbody*2/1000,np.mean(frame.body,axis = 0) + frame.xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
# Plotters.scatter3d(fig, lt20p,'magenta',5,'x') 
# Plotters.scatter3d(fig,bod_ax_top,'black',3,'body',show_colorbar = False)

# Plotters.scatter3d(fig, np.atleast_2d(frame.interest_on_xbody),'black',10,'inter_on_body',mode = 'markers+lines') 
# Plotters.scatter3d(fig, frame.rotated_points_3d[16:,:],'black',10,'inter',mode = 'markers+lines') 


# fig.show()

# ax = None
# ax = Plotters.plot_projections(frame.interest_points_3d[:,:],frame.frames,color = 'magenta',ax = ax, size = 5)
# ax = Plotters.plot_projections(frame.gaussian_closest_to_interest_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)
# # ax = Plotters.plot_projections(frame.body_interest_gaussian_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)

# ax = None
# ax = Plotters.plot_projections(frame.right_wing_ew[:,:],frame.frames,color = 'red',ax = ax, size = 5)
# ax = Plotters.plot_projections(frame.left_wing_ew[:,:],frame.frames,color = 'blue',ax = ax, size = 5)


AttributeError: 'FlyOutput' object has no attribute 'dist_points'

In [ ]:
dot_on_le = np.dot(frame.right_wing_direction,frame.right_wing_le.T)
length = (np.max(dot_on_le) - np.min(dot_on_le))
lt20p = dot_on_le[dot_on_le > length*0.2,:]

0.002344680935456922

In [39]:
idx_origin = np.argmin([np.min(np.dot(frame.body[idx_closest,:],r_line_points.T)) for idx_closest in idx_closest])
idx_origin

89

In [ ]:
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)

plt.figure(),plt.plot(pts_on_nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
plt.figure()
plt.plot((pts_on_nrml - mean)/std)


In [11]:
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)
plt.figure()
plt.plot(points_2d[:,0],points_2d[:,1])

plt.figure()
plt.plot(points_2d)

In [ ]:

wing_gs,interest_rw,interest_lw = frame.zsocre_ol_calc_indices()


In [21]:
from scipy.signal import savgol_filter


frame = frames_list[-1]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)


x = savgol_filter(points_2d[:,0], 15, 2)
y = savgol_filter(points_2d[:,1], 15, 2)


plt.scatter(points_2d[:,0],points_2d[:,1])

plt.scatter(x,y)

In [ ]:
frame = frames_list[-1]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)



wing_bound = wing_bound[indices_wing_bound]
pts_to_fit = [wing_bound[k:k+3] for k in range(0,wing_bound.shape[0],3)]

fit = []
for pts in pts_to_fit[:-1]: 

    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning
    p = np.polyfit(t, pts, 2)
    t_fit = np.linspace(t[0], t[-1], 1000)
    fit_xyz = np.vstack([np.polyval(p, t_fit) for p in p.T]).T
    fit.append(fit_xyz)
fit = np.vstack(fit)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot(wing_bound[:,0], wing_bound[:,1], wing_bound[:,2], 'ro', label='Original points')
ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
ax.legend()
plt.show()

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned



In [26]:
fit.shape

(21000, 3)

In [ ]:
from math import atan2
normal_to_wing = np.cross(frame.right_wing_span,frame.right_wing_chord)

wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))



frameidx = 4
frame = frames_list[frameidx]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
interest = frames_list[frameidx].rotated_points_3d[frame.interest_right_wing_boundry,:]



points_2d = Utils(wing_bound, np.mean(wing_bound,axis = 0), frame.right_wing_span, frame.right_wing_chord)
indices = Utils.rotational_sort(points_2d, np.mean(points_2d,axis = 0), clockwise=True)
# indices2 = rotational_sort(points_2d, np.mean(points_2d,axis = 0), clockwise=True)


wing_bound
fig = go.Figure()
# Plotters.scatter3d(fig,frame.right_wing,'red',3,'body',show_colorbar = False)
# Plotters.scatter3d(fig,interest[indices,:],'black',3,'interest',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,wing_bound[indices2,:],'blue',3,'gs',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,np.atleast_2d(frame.right_wing_origin),'blue',10,'gs',show_colorbar = False,mode='markers+lines')

fig.show()


In [30]:
from math import atan2

def argsort(seq):
    #http://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python/3382369#3382369
    #by unutbu
    #https://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python 
    # from Boris Gorelik
    return sorted(range(len(seq)), key=seq.__getitem__)

def rotational_sort(list_of_xy_coords, centre_of_rotation_xy_coord, clockwise=True):
    cx,cy=centre_of_rotation_xy_coord
    angles = [atan2(x-cx, y-cy) for x,y in list_of_xy_coords]
    indices = argsort(angles)
    # if clockwise:
    #     return [list_of_xy_coords[i] for i in indices]
    # else:
    #     return [list_of_xy_coords[i] for i in indices[::-1]]
    return indices

frameidx = 4
frame = frames_list[frameidx]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
interest = frames_list[frameidx].rotated_points_3d[frame.interest_right_wing_boundry,:]
indices = rotational_sort(interest[:,[1,2]], np.mean(interest[:,[1,2]],axis = 0), clockwise=True)
indices2 = rotational_sort(wing_bound[:,[1,2]], np.mean(wing_bound[:,[1,2]],axis = 0), clockwise=True)


wing_bound
fig = go.Figure()
# Plotters.scatter3d(fig,frame.right_wing,'red',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,interest[indices,:],'black',3,'interest',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,wing_bound[indices2,:],'blue',3,'gs',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,np.atleast_2d(frame.right_wing_origin),'blue',10,'gs',show_colorbar = False,mode='markers+lines')

fig.show()


In [59]:
import numpy as np
import matplotlib.pyplot as plt

# Your 3D points (N x 3)
points = interest[indices,:]
# points = wing_bound[indices2,:]

pts_to_fit = [points[k:k+3] for k in range(0,points.shape[0],3)]
# pts_to_fit = [points[k:k+5] for k in range(0,points.shape[0],5)]


fit = []
for pts in pts_to_fit[:-1]: 
    # Step 1: Generate parameter t (cumulative distance)
    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning

    # Step 2: Fit a 2nd degree polynomial separately for x(t), y(t), z(t)
    degree = 2
    px = np.polyfit(t, pts[:,0], degree)
    py = np.polyfit(t, pts[:,1], degree)
    pz = np.polyfit(t, pts[:,2], degree)

    # To evaluate the fitted curve:
    t_fit = np.linspace(t[0], t[-1], 1000)
    x_fit = np.polyval(px, t_fit)
    y_fit = np.polyval(py, t_fit)
    z_fit = np.polyval(pz, t_fit)
    fit.append(np.vstack((x_fit, y_fit, z_fit)))
fit = np.hstack(fit).T
# Plotting
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.plot(points[:,0], points[:,1], points[:,2], 'ro', label='Original points')
# ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
# ax.legend()
# plt.show()


In [60]:
import numpy as np
import matplotlib.pyplot as plt

# Your 3D points (N x 3)
points = interest[indices,:]
points = wing_bound[indices2,:]

pts_to_fit = [points[k:k+3] for k in range(0,points.shape[0],3)]
# pts_to_fit = [points[k:k+5] for k in range(0,points.shape[0],5)]


fit2 = []
for pts in pts_to_fit[:-1]: 
    # Step 1: Generate parameter t (cumulative distance)
    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning

    # Step 2: Fit a 2nd degree polynomial separately for x(t), y(t), z(t)
    degree = 2
    px = np.polyfit(t, pts[:,0], degree)
    py = np.polyfit(t, pts[:,1], degree)
    pz = np.polyfit(t, pts[:,2], degree)

    # To evaluate the fitted curve:
    t_fit = np.linspace(t[0], t[-1], 1000)
    x_fit = np.polyval(px, t_fit)
    y_fit = np.polyval(py, t_fit)
    z_fit = np.polyval(pz, t_fit)
    fit2.append(np.vstack((x_fit, y_fit, z_fit)))
fit2 = np.hstack(fit2).T
# Plotting
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.plot(points[:,0], points[:,1], points[:,2], 'ro', label='Original points')
# ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
# ax.legend()
# plt.show()


c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarni

In [61]:

def dist_points(x1,x2):
    return np.sqrt(np.sum((x1 - x2)**2, axis = 1))

gaussian_closest_to_interest = np.vstack((fit2[np.argmin(dist_points(fit2,point)),:] for point in fit))
fitted_closest_to_gauss = np.vstack((fit[np.argmin(dist_points(fit,point)),:] for point in fit2))


plt.figure()
plt.hist(1000*dist_points(gaussian_closest_to_interest,fit))

plt.figure()
plt.hist(1000*dist_points(fitted_closest_to_gauss,fit2))

C:\Users\Roni\AppData\Local\Temp\ipykernel_14272\387352763.py:4: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.

C:\Users\Roni\AppData\Local\Temp\ipykernel_14272\387352763.py:5: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.



(array([3399., 5420., 5358., 4233., 2123.,  666.,  294.,  169.,  161.,
         177.]),
 array([0.00934659, 0.03537276, 0.06139894, 0.08742511, 0.11345128,
        0.13947745, 0.16550363, 0.1915298 , 0.21755597, 0.24358214,
        0.26960831]),
 <BarContainer object of 10 artists>)

In [ ]:


def dist_points(x1,x2):
    return np.sqrt(np.sum((x1 - x2)**2, axis = 1))

gaussian_closest_to_interest = np.vstack((fit[np.argmin(dist_points(fit,point)),:] for point in wing_bound))


    def closest_point_to_interest_boundary(self,wing_boundary,points):   

        gaussian_closest_to_interest = np.vstack((wing_boundary[np.argmin(self.dist_points(wing_boundary,point)),:] for point in points))
        gaussian_closest_to_interest_ew = (self.ew_to_lab.T @ np.vstack(gaussian_closest_to_interest).T).T
        dist_gaus_interest = self.dist_points(gaussian_closest_to_interest[1:,:],gaussian_closest_to_interest[0:-1,:])
        dist_interest = self.dist_points(points[1:,:],points[0:-1,:])   
        return  gaussian_closest_to_interest,gaussian_closest_to_interest_ew,dist_gaus_interest,dist_interest
